In [13]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/neurodiscoveryai/Documents/deidentification_df/deidentification/deIdentification/')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

import pandas as pd
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel,IgnoreRowsDeIdentificaiton
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task, _get_default_table_details_for_ui

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [8]:
def fix_reference_value(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "PATIENT_ID"
    ]
    enc_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "ENCOUNTER_ID"
    ]
    print(pid_col)
    print(enc_col)
    patient_id = pid_col[0] if len(pid_col)>0 else None
    enc_id = enc_col[0] if len(enc_col)>0 else None
    table.table_details_for_ui['reference_enc_id_column'] = enc_id
    table.table_details_for_ui['reference_patient_id_column'] = patient_id
    table.save()

In [9]:
tables = ["ELIGIBILITYTRACK"]

In [10]:
tables_objs = TableDetailsModel.objects.filter(table_name__in=tables).order_by('rows_count')
totalrun = 0
for tableobj in tables_objs:
    #tableobj = TableDetailsModel.objects.get(table_name=table)
    Task.objects.filter(arguments__table_id=tableobj.id).delete()
    #if tableobj.table_status in [2]:
    #    continue
    fix_reference_value(tableobj.id)
    tableobj.refresh_from_db()
    tasks, chain = create_deidentification_task(tableobj)
    totalrun += 1
    print(tableobj.rows_count)

2025-07-10 06:17:17,850 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2025-07-10 06:17:17,856 - deIdentification.nd_logger - WARNING - Table ELIGIBILITYTRACK does not exist and cannot be dropped.
2025-07-10 06:17:17,856 - deIdentification.nd_logger - INFO - Dropping ignore rows for ELIGIBILITYTRACK, postgres


['PATIENTID', 'ALTPATIENTID']
[]
10143922


In [55]:
from django.db.models import Count

grouped_tasks = Task.all_objects.filter(status=0).values('arguments__table_id').annotate(count=Count('id'))
#print(list(counts))

table_ids = [entry['arguments__table_id'] for entry in grouped_tasks if entry['arguments__table_id'] is not None]

table_info = TableDetailsModel.objects.filter(id__in=table_ids).values('id', 'table_name')
table_map = {entry['id']: entry['table_name'] for entry in table_info}

#print(table_info)
#print(table_map)

result = []
for entry in grouped_tasks:
    table_id = entry['arguments__table_id']
    count = entry['count']
    
    table_name = table_map.get(table_id, 'Unknown')
    result.append({'table_id': table_id, 'table_name': table_name, 'count': count})


total_tasks=0
total_tables = []
for r in result:
    total_tasks += r.get('count', 0)
    total_tables.append(r.get('table_name'))
    print("Tasks Left: ", r)
print("Total Tasks:", total_tasks)
print("Total Tables:", len(total_tables))

Tasks Left:  {'table_id': 95, 'table_name': 'CLIENT', 'count': 1}
Tasks Left:  {'table_id': 227, 'table_name': 'DOCUMENTDIAGNOSIS', 'count': 1}
Tasks Left:  {'table_id': 244, 'table_name': 'ELIGIBILITYTRACK', 'count': 40}
Total Tasks: 42
Total Tables: 3


In [39]:
IgnoreRowsDeIdentificaiton.objects.filter(table_name='ELIGIBILITYTRACK').first().row

{'NPI': 'None',
 'DEBUG': 'None',
 'STATE': 'F',
 'RULEID': 'None',
 'TESTYN': 'N',
 'NPITYPE': 'None',
 'PCPNAME': 'None',
 'IDNUMBER': 'None',
 'USERNAME': 'autoelig',
 'CONTEXTID': '25810',
 'CREATEDBY': 'autoelig',
 'DELETEDBY': 'None',
 'PATIENTID': 'None',
 'EDIBATCHID': 'None',
 'ORIGINALID': '827330',
 'PROVIDERID': 'None',
 'UNVERIFIED': 'None',
 'CONTEXTNAME': 'NC - RALEIGH NEUROLOGY ASSOCIATES, P.A.',
 'ELAPSEDTIME': 'None',
 'LASTUPDATED': '2025-02-20 17:45:00',
 'ALTPATIENTID': 'None',
 'DEPARTMENTID': '20.0',
 'POLICYNUMBER': 'None',
 'RECEIVERNAME': 'BCBS_NC',
 'REFERENCEKEY': 'None',
 'RULELOCATION': 'None',
 'APPOINTMENTID': '1137388.0',
 'INQUIRYREASON': 'NIGHTLY',
 'ISSUECATEGORY': 'None',
 'MESSAGERESULT': '7.3',
 'SYNCHRONOUSYN': 'None',
 'TRANSACTIONYN': 'N',
 'ELIGIBLERESULT': 'None',
 'HIXGRACEPERIOD': 'None',
 'LASTMODIFIEDBY': 'autoelig',
 'OTHERPAYERFLAG': 'None',
 'PROVIDERSOURCE': 'None',
 'ACKNOWLEDGEMENT': 'None',
 'CREATEDDATETIME': '2023-02-20 07:25:01'

In [53]:
db, _ = DbDetailsModel.objects.get_or_create(db_name="postgres")
t = TableDetailsModel.objects.get(table_name = 'ELIGIBILITYTRACK', db=db)
config = t.table_details_for_ui
config



{'batch_size': 1000,
 'ignore_rows': {},
 'columns_details': [{'is_phi': False,
   'mask_value': '',
   'column_name': 'CONTEXTID',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': True,
   'mask_value': 'FACILITYNAME',
   'column_name': 'CONTEXTNAME',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': 'MASK',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'CONTEXTPARENTCONTEXTID',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'ELIGIBILITYTRACKID',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'i

In [40]:
for item in config['columns_details']:
    if not item['is_phi']:
        try:
            print(item['column_name'], item['de_identification_rule'],item['mask_value'])
        except Exception as e:
            print(e)

CONTEXTID  
CONTEXTPARENTCONTEXTID  
ELIGIBILITYTRACKID  
ORIGINALID  
STATE  
ATTEMPTSREMAINING  
ELIGIBILITYCLASS  
RECEIVERNAME  
INFORMATIONTYPE  
PROVIDERID  
REFERENCEKEY  
TRANSACTIONYN  
INQUIRYREASON  
ACKNOWLEDGEMENT  
SYNCHRONOUSYN  
ELIGIBILITYWARNING  
ELAPSEDTIME  
TESTYN  
DEPARTMENTID  
APPOINTMENTID  
INSURANCEPACKAGEID  
RELATIONSHIPTOINSUREDID  
INFORMATIONSOURCENAME  
PCPNAME  
OUTGOINGANSIVERSIONID  
INCOMINGANSIVERSIONID  
ELIGIBILITYSEARCHCRITERIAID  
ENROLLMENTPERIODID  
PROVIDERSOURCE  
NPITYPE  
NPI  
ITERATIVEMESSAGEMAPID  
SERVICETYPECODE  
ISSUECATEGORY  
PARENTELIGIBILITYTRACKID  
ELIGIBILITYBATCHID  
RULELOCATION  
RULEID  
PATIENTMATCHSCORE  
ELIGIBLERESULT  
UNVERIFIED  
INCORRECTPACKAGE  
DEMOGRAPHICMISMATCH  
HIXGRACEPERIOD  
INCORRECTPATIENTPHI  
EDIBATCHID  
LIMITEDCOVERAGEYN  
CACHEDELIGIBILITYTRACKID  
OTHERPAYERFLAG  
USECACHINGLOGICFLAG  
ELIGIBLEINSTITUTIONALFLAG  
INSURANCEPRODUCTTYPECODE  
CREATEDBY  
LASTMODIFIEDBY  
DELETEDBY  
'mask_value'